In [ ]:
using NPZ, Healpix, Statistics, Plots

# Load Planck temperature map saved from Python
I = npzread("Tpatch_10deg.npy")
I_julia = Array(I)

# Define Healpix resolution (instead of plain nside)
res = Healpix.Resolution(2048)

# Helper: degrees → radians
deg2rad(x) = x * (π / 180)

"""
Extract a flat patch from a Healpix map in RING ordering.
"""
function healpix_patch(I_julia, res; lon0=0.0, lat0=0.0, size_deg=10.0, res_arcmin=2.0)
    npix = Int(round(size_deg * 60 / res_arcmin))
    patch = Array{Float32}(undef, npix, npix)
    for i in 1:npix, j in 1:npix
        dl = (i - (npix + 1)/2) * res_arcmin / 60
        db = (j - (npix + 1)/2) * res_arcmin / 60
        lon = lon0 + dl
        lat = lat0 + db
        θ = deg2rad(90 - lat)
        ϕ = deg2rad(mod(lon, 360))
        z = cos(θ)
        p = Healpix.zphi2pixRing(res, z, ϕ)
        patch[i, j] = I_julia[p + 1]   # Julia indexing
    end
    return patch
end

# North galactic cap (cleaner CMB)
Tpatch_high = healpix_patch(I_julia, res; lon0=0.0, lat0=60.0, size_deg=10.0, res_arcmin=2.0)
heatmap(Tpatch_high'; aspect_ratio=1, title="Planck T patch (lat=+60°)")


In [ ]:

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
# Load the 3 maps (I, Q, U)
I, Q, U = hp.read_map("COM_CMB_IQU-smica_2048_R3.00_full.fits", field=(0, 1, 2))
np.save("Tpatch_10deg.npy",I)
print(I.shape)   # (50331648,)

# Plot full-sky T (I) map
hp.mollview(I, title="Planck SMICA T map (NSIDE=2048)", unit="K")
plt.show()


In [ ]:
using CMBLensing, Plots, Statistics

# --------------------------------------------------------
# 1. Your Planck SMICA temperature patch
# --------------------------------------------------------
N = size(Tpatch, 1)           # e.g. 300
θpix = 2.0                    # arcmin per pixel
Tpatch .-= mean(Tpatch)       # remove monopole

# --------------------------------------------------------
# 2. Build a CMBLensing FlatMap
# --------------------------------------------------------
mapdata = Float32.(permutedims(Tpatch, (2,1)))  # x-columns, y-rows
I_flat = FlatMap(mapdata; θpix=θpix)

# --------------------------------------------------------
# 3. Create a simulated φ field (for testing)
# --------------------------------------------------------
(; ds, f, ϕ) = load_sim(
    θpix  = θpix,
    Nside = N,
    T     = Float32,
    pol   = :I,
)


# --------------------------------------------------------
# 5. Lens / delens operations
# --------------------------------------------------------
L = LenseFlow(ϕ)

# Delensing (inverse lensing)
f_unlensed = L \ f_lensed

# Re-lensing to check consistency
f_relensed = L * f_unlensed

# --------------------------------------------------------
# 6. Plot with Plots.jl
# --------------------------------------------------------
plt = Plots.plot(layout=(1,3), size=(1200,400))
Plots.heatmap!(plt[1], f_lensed[:Ix]', aspect_ratio=1, title="Input (Planck lensed)")
Plots.heatmap!(plt[2], f_unlensed[:Ix]', aspect_ratio=1, title="Delensed (approx)")
Plots.heatmap!(plt[3], (f_relensed[:Ix]' .- f_lensed[:Ix]') .* 1e6, aspect_ratio=1, title="Residuals (µK)")
display(plt)

In [ ]:
using MuseInference, CMBLensing, Statistics

# Your existing patch
θpix = 2.0
I_flat = FlatMap(Float32.(permutedims(Tpatch, (2,1))); θpix=θpix)

# Optional: apodization
mask = apodize_mask(I_flat, taper_deg=1.0)   # simple cosine window, defined in utils.jl

# Build the dataset descriptor (like in CMBLensing)
ds = DataSet(I=I_flat, mask=mask)

# Load theory spectra (Cl_TT, Cl_EE, Cl_PP, etc.)
theory = load_theory("planck2018_lensedCls.dat")

# Run MAP lensing reconstruction
recon = reconstruct_phi(ds, theory; maxiter=50, tol=1e-4)

# Extract outputs
phi_map = recon.ϕ          # reconstructed lensing potential
kappa_map = recon.κ        # convergence if available

Plots.heatmap(phi_map[:Ix]', aspect_ratio=1, title="Reconstructed φ (MUSEInference)")


In [ ]:
using PythonCall

# Import Python CAMB
camb = pyimport("camb")

pars = camb.CAMBparams()
pars.set_cosmology(H0=67.4, ombh2=0.0224, omch2=0.12, mnu=0.06)
pars.InitPower.set_params(As=2.1e-9, ns=0.965)
pars.set_for_lmax(lmax=4000, lens_potential_accuracy=1)

results = camb.get_results(pars)
cls = results.get_cmb_power_spectra(CMB_unit="muK")["unlensed_scalar"]   # TT, EE, BB, TE
cls_lensed = results.get_cmb_power_spectra(CMB_unit="muK")["total"]      # includes lensing

ϕϕ = results.get_lens_potential_cls(lmax=4000)[:,0]                      # C_ℓ^{φφ}
l = collect(0:length(ϕϕ)-1)
CTT = cls[:,1]    # C_ℓ^{TT}
CEE = cls[:,2]
CBB = cls[:,3]
CTE = cls[:,4]
Cf  = importCls((; ell=l, CTT=CTT, CEE=CEE, CBB=CBB, CTE=CTE))
Cϕ  = importCls((; ell=l, Cpp=ϕϕ))

beamFWHM = 5.0              # SMICA beam (arcmin)
μKarcminT = 45.0            # approximate noise
Cf  = importCls(planck_bestfit)   # or default_Cf()
Cϕ  = default_Cϕ()
Cn  = white_noise_Cn(N, θpix; μKarcminT=μKarcminT, beam=GaussianBeam(beamFWHM))
mask = apodize(circular_mask(N; radius_deg=4.5); apodization_deg=0.5)
ds = DataSet(; d = Field(Ix=my_T_map), Cf=Cf, Cϕ=Cϕ, Cn=Cn, B=GaussianBeam(beamFWHM), M=PixelMask(mask), L=LenseFlow(10))


In [ ]:
using CMBLensing

θpix = 2.0                     # arcmin / pixel (your patch)
N    = size(Tpatch,1)

# (A) signal & φ covariances
Cf  = default_Cf(; pol=:I)
Cϕ  = default_Cϕ()

# (B) instrument model (Planck-like ballpark)
beamFWHM   = 5.0               # arcmin (SMICA ~5′; adjust if you know it)
μKarcminT  = 45.0              # white noise level in T (ballpark; set your true value)

Cn = white_noise_Cn(N, θpix; μKarcminT=μKarcminT, beam=GaussianBeam(beamFWHM))

# (C) mask & bandpass (help conditioning and boundary artefacts)
M = PixelMask(apodize(ones(Float32,N,N); apodization_deg=0.5))   # flat patch → apodize edges
B = GaussianBeam(beamFWHM)
G = LowPass(3000)   # optional band-limit; helps speed

# (D) wrap your map as a Field
# CMBLensing expects x→columns, y→rows; transpose like you did for plots:
Ix_map = FlatMap(Float32.(permutedims(Tpatch,(2,1))); θpix=θpix)
d      = Field(Ix = Ix_map)

# (E) build the dataset
ds = DataSet(; d, Cf, Cϕ, Cn, B, M, G, L=LenseFlow(10))


Attempts to use MAP_joint with qe as intial input

In [ ]:
# print some grid facts
proj = ϕ.metadata
@unpack Δℓx, Δℓy, nyquist, ℓmag = proj
@show Δℓx Δℓy nyquist
println("ℓ range in map grid: ", extrema(vec(ℓmag)))


In [1]:
using StatsBase
using CMBLensing: Fourier, Cℓs

"""
    get_Cℓ_cross(f1::LambertS0, f2::LambertS0; Δℓ=50)

Flat-sky cross-power spectrum Cℓ(f1, f2) using CMBLensing rFFT layout.
Returns a Cℓs object (ℓ, Cℓ).
"""
function get_Cℓ_cross(f1, f2; Δℓ=50)
    F1 = Fourier(f1)               # CMBLensing rFFT object
    F2 = Fourier(f2)

    Pk = real.(F1.arr .* conj.(F2.arr)) .* F1.metadata.λ_rfft ./ (F1.metadata.Nx * F1.metadata.Ny)

    ℓ = vec(F1.metadata.ℓmag)
    P = vec(Pk)

    edges = 0:Δℓ:maximum(ℓ)

    H  = fit(Histogram, ℓ, Weights(P), edges)
    Hw = fit(Histogram, ℓ, Weights(fill(1.0,length(ℓ))), edges).weights

    Cℓ = H.weights ./ max.(Hw, 1)
    ℓc = (edges[1:end-1] .+ edges[2:end]) ./ 2

    return Cℓs(ℓc, Cℓ)
end


┌ Error: LLVM extensions library unavailable for your platform:
│   aarch64-apple-darwin-libgfortran5-cxx11-llvm_version+18-julia_version+1.12.1
│ LLVM.jl will not be functional.
│ 
│ If you are using a custom version of LLVM, try building a
│ custom version of LLVMExtra_jll using `deps/build_local.jl`
└ @ LLVM ~/.julia/packages/LLVM/bzSzE/src/LLVM.jl:103


get_Cℓ_cross

In [2]:
CϕϕJ = get_Cℓ_cross(ϕ, ϕJ; Δℓ=80)
CϕJϕJ = get_Cℓ_cross(ϕJ, ϕJ; Δℓ=80)

mask = .!isnan.(CϕϕJ.Cℓ) .& (CϕϕJ.Cℓ .> 0)

loglog(CϕϕJ.ℓ[mask].^4 .* CϕϕJ.Cℓ[mask], label="cross(ϕ_true × ϕ_MAP)")
loglog(Cℓ.total.ϕϕ.ℓ.^4 .* Cℓ.total.ϕϕ.Cℓ, "k", label="theory")
xlabel(raw"$\ell$")
ylabel(raw"$\ell^4 C_\ell$")
legend()
PythonPlot.show()

# Compute spectra
Cϕϕ_true = get_Cℓ_cross(ϕ, ϕ; Δℓ=80)         # auto: true φ × true φ
Cϕϕ_cross = get_Cℓ_cross(ϕ, ϕJ; Δℓ=80)       # cross: true φ × MAP φ
Cϕϕ_rec  = get_Cℓ_cross(ϕJ, ϕJ; Δℓ=80)       # auto: MAP φ × MAP φ

# Extract ℓ and Cℓ vectors
ℓ = Cϕϕ_cross.ℓ
Cx = Cϕϕ_cross.Cℓ        # cross
Ct = Cϕϕ_true.Cℓ         # true auto
Cr = Cϕϕ_rec.Cℓ          # reconstruction auto

# Compute correlation coefficient per ℓ-bin
ρℓ = Cx ./ sqrt.(Ct .* Cr)
plt.figure(figsize=(7,4))
plt.semilogx(ℓ, ρℓ)
plt.axhline(1, color="k", linestyle="--")
plt.xlabel(L"\ell")
plt.ylabel(L"\rho_\ell")
plt.title(L"\rho_\ell = \frac{C_\ell^{\phi \hat{\phi}}}{\sqrt{C_\ell^{\phi\phi} C_\ell^{\hat{\phi}\hat{\phi}}}}")

plt.ylim(0, 1.1)
plt.show()


LoadError: UndefVarError: `ϕ` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
using NPZ

# helper to unwrap Cf, Cphi, Cn
getdiag(x) = x isa Diagonal ? x.diag :
             x isa CMBLensing.ParamDependentOp ? x.op.diag :
             error("Unknown covariance operator type")

# convert string metadata → byte arrays (NPZ compatible)
asbytes(s::String) = Vector{UInt8}(codeunits(s))

npzwrite("sim_maps.npz", Dict(
    # CMB MAPS
    "f_unlensed" => f,
    "f_lensed"   => f̃,
    "phi"        => ϕ,

    # POWER SPECTRA (CAMB output)
    "ell"        => Cℓ.total.ϕϕ.ℓ,
    "Cl_phi"     => Cℓ.total.ϕϕ.Cℓ,
    "Cl_TT"      => Cℓ.total.TT.Cℓ,
    "Cl_n"       => Cℓn.TT.Cℓ,

    # COVARIANCES used in MAP
    "Cf_diag"    => getdiag(ds.Cf),
    "Cphi_diag"  => getdiag(ds.Cϕ),
    "Cn_diag"    => getdiag(ds.Cn),


    "theta_pix_arcmin" => θpix,
    "Nside" => Nside,
    "seed"  => 3,
    "dtype" => asbytes(string(T)),
    "pol"   => asbytes(string(pol)),
))

println("Saved to sim_maps.npz (maps + Cℓ + covariances + metadata)")
